# CEBRA-Lens Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdaptiveMotorControlLab/CEBRA-lens/blob/main/Notebooks/Demo-Notebook-Allen.ipynb)

<!-- #fix the link later -->

### Download dataset

This notebook will demo how to use CEBRA-Lens on the Allen visual dataset. To download the allen data please follow the [link](https://figshare.com/s/60adb075234c2cc51fa3?file=36869049). Make sure the data is placed in a folder `/data`. As a prerequisite for the notebook you should have models already trained. If not, you can refer to the [Generate Models notebook](ModelGenerator.ipynb).

In [ ]:
import cebra_lens as lens

### Load data and models

Data is loaded here with a utils functions for the Allen dataset. Models are loaded by using the name of the folder where they are located as well as the folder path. It will load all the models ending with .pt located in the path, thus these models should be saved as **torch** and not as sklearn. This was done due to easier loading on the CPU compared to the sklearn

The function model_loader allows for inputing labels for models which allow for grouping based on prefered parameters. The labels are given as a dictionary where the keys correspond to the model file name and the value to the prefered label for that model.

In [ ]:
model_path = "/content/drive/MyDrive/mathis/FinalModels/VISION/offset10"

Example of labels dictionary:

In [ ]:
labels = {'allen_single_session_mouse4_0k_UT_torch':'single_UT',
          'allen_multi_session_10k_3_torch':'multi_TR',
          'allen_multi_session_10k_2_torch':'multi_TR',
          'allen_single_session_mouse4_10k_0_torch':'single_TR',
          'allen_single_session_mouse4_10k_1_torch':'single_TR',
          'allen_multi_session_10k_4_torch':'multi_TR',
          'allen_multi_session_10k_0_torch':'multi_TR',
          'allen_multi_session_10k_1_torch':'multi_TR',
          'allen_single_session_mouse4_10k_3_torch':'single_TR',
          'allen_single_session_mouse4_10k_2_torch':'single_TR',
          'allen_multi_session_0k_UT_torch':'multi_UT',
          'allen_single_session_mouse4_10k_4_torch':'single_TR'}

In [ ]:
models = lens.model_loader(model_path, labels=labels)

To load data make sure you are in the directory where the `/data` folder is located

In [ ]:
session_id = 3
train_datas, valid_datas, discrete_labels_train, discrete_labels_val = (
    lens.utils_allen.get_single_session_datasets()
)
#this works fine

train_data = train_datas[session_id].neural
test_data = valid_datas[session_id].neural
train_label = discrete_labels_train[session_id]
test_label = discrete_labels_val[session_id]

### Model decoding

To verify the model performance the decode_models function can be called. This will give a dictionnary with each key being the name of the model "single_UT" "multi_TR" ... and plot the performance accordingly. This is flexible and can thus take many different models.

In [ ]:
from lens.quantification import *
from tqdm import tqdm

In [ ]:
dataset_label = "visual"

In [ ]:
decoding_models = decoding.DecodeModel(train_data, train_label, test_data, test_label, session_id, dataset_label)

In [ ]:
multi_models = multibase.MultiModel(decoding_models)

In [ ]:
results_dict = multi_models.compute(models)

In [ ]:
fig = decoding_models.plot(results_dict=results_dict, palette="cool")

### Model decoding by layer

In [ ]:
dataset_label = "visual"
layer_type = "conv"

In [ ]:
multi_per_layer = decoding.Decoding(train_data, train_label, test_data, test_label, session_id, dataset_label)
multi_models = multibase.MultiModel(multi_per_layer)
results_dict = multi_models.compute(models)

In [ ]:
fig = multi_per_layer.plot(results_dict=results_dict)

### Retrieve and compare embeddings

In [ ]:
activations = {}
activations = lens.activations.get_activations_models(
    models=models,
    data=train_data,
    session_id=session_id,
    activations=activations,
    layer_type=layer_type,
)

In [ ]:
activations_dict = lens.activations.process_activations(activations)

#### Plot activations for one model

In [ ]:
instance = 0
label = "multi_TR"

In [ ]:
plot_data = activations_dict[label][instance]

In [ ]:
fig = lens.plot_activations(
    train_data,
    plot_data,
    title="Single Trained activations across layers",
    figsize=(7, 14),
)

#### Compare embedddings of two models across layers

In [ ]:
model1 = activations_dict["single_TR"][0]
model2 = activations_dict["multi_TR"][0]

In [ ]:
fig = lens.compare_embeddings_layers(
    model1,
    model2,
    labels=train_label,
    dataset_label=dataset_label,
    sample_plot=model2[0].shape[1],
    comparison_labels=("CEBRA embeddings", ["Single", "Multi"]),

#### Compare embeddings after tSNE dimensionality reduction 

In [ ]:
tsne_model = tsne.Tsne(num_samples_tSNE)
multi_models = multibase.MultiModel(tsne_model)
tSNE_dict = multi_models.compute(activations_dict)

In [ ]:
fig = lens.compare_embeddings_layers(
    tSNE_dict["single_TR"][0],
    tSNE_dict["multi_TR"][0],
    labels=train_label,
    dataset_label=dataset_label,
    sample_plot=num_samples_tSNE,
    comparison_labels=("tSNE", ["Single", "Multi"]),
)

### CKA Analysis

To compute the CKA you can chose the desired comparisons and it will create the CKA matrix accordingly. These comparisons are a list of tuples. Where the tuples comprise of model labels

In [ ]:
comparisons = [
        ("single_UT", "single_TR"),
        ("multi_UT", "multi_TR"),
        ("single_TR", "multi_TR"),
        ("single_TR", "single_TR"),
        ("multi_TR", "multi_TR"),
    ]

In [ ]:
cka_matrices = {}
for comparison in comparisons:
        cka_multi = CKA(comparison)
        cka_matrix = cka_multi.compute(activations_dict)
        cka_matrices[f"{comparison[0]}_v_{comparison[1]}"] = cka_matrix

In [ ]:
fig = lens.plot_cka_heatmaps(
    cka_matrices=cka_matrices,
    annot=False,
)

### RDM (Representational Dissimilarity Matrix) Analysis

RDM Matrices are huge to save (20Go) but not so long to compute. Be careful when saving it, comment if needed or at least delete it when no longer needed it. rdm_dict will have the same format as activation_dict. It computes the RDM for the whole activation dataset, but smaller datasets can be inputted. It also has the possibility to correlate it to Oracle RDM.

In [ ]:
First, we show an example of an RDM computed on all layers of a model instance. We can plot the desired layers.

In [ ]:
rdm_single = RDM(
    data=train_data,
    label=train_label,
    metric="correlation",
    bool_oracle=False,
)
multi_rdm_corr = rdm_single.compute(activations_dict["multi_TR"][0])

In [ ]:
fig = rdm_single.plot(
        [multi_rdm_corr[0][0]],
        ["Layer 1"],
        metric="Correlation",
    )
fig2 = rdm_single.plot(
        [multi_rdm_corr[1][0], multi_rdm_corr[-1][0]],
        ["Layer 2", "Output Layer"],
        metric="Correlation",
    )
fig3 = rdm_single.plot(
        [
            multi_rdm_corr[0][0],
            multi_rdm_corr[1][0],
            multi_rdm_corr[2][0],
            multi_rdm_corr[3][0],
            multi_rdm_corr[-1][0],
        ],
        ["Layer 1", "Layer 2", "Layer 3", "Layer 4", "Output Layer"],
        metric="Correlation",
        figsize=(20, 8),
    )

We can compute the analysis on all instances and all models by passing the dictionary activations fully.

In [ ]:
rdm_models = multibase.MultiModel(
    lens.quantification.RDM(
        data=train_data,
        label=train_label,
        metric="correlation",
        bool_oracle=True,
        dataset_label=dataset_label,
    ))
rdm_dict = rdm_models.compute(activations_dict)

In [ ]:
fig = lens.plot_rdm_correlation(rdm_dict)

### Distance metric analysis

A final embedding analysis can be done using cebra_lens: a distance analysis. They all utilise the binning as described in RDM (c.f. `cebra_lens.quantification.misc.discrete_binning()`).
Three distances can be computed:
- inter-bin -> between the bins (irrespective of repetitions)
- intra-bin -> inside of each bin
- inter-repetition -> distance between each repetition by computing centroid of each bin at each repetition

In [ ]:
metric = "cosine"

#### Inter-repetition

In [ ]:
multimodel_interrep = lens.quantification.mutlibase.MultiModel(
    lens.quantification.distance.Distance(
        data=train_data,
        label=train_label,
        dataset_label=dataset_label,
        metric=metric,
        distance_label="interrep",
    ))
interrep_dict = multimodel_interrep.compute(activations_dict)

In [ ]:
fig = multimodel_interrep.plot(
    interrep_dict,
    title="Inter-representation distance",
)

#### Intra-bin

In [ ]:
multimodel_intrabin = lens.quantification.mutlibase.MultiModel(
    lens.quantification.distance.Distance(
        data=train_data,
        label=train_label,
        dataset_label=dataset_label,
        metric=metric,
        distance_label="intrabin",
    ))
intrabin_dict = multimodel_intrabin.compute(activations_dict)

In [ ]:
fig = multimodel_intrabin.plot(
    intrabin_dict,
    title="Intra-bin distance",
)

#### Inter-bin

In [ ]:
multimodel_interbin = lens.quantification.multibase.MultiModel(
    lens.quantification.distance.Distance(
        data=train_data,
        label=train_label,
        dataset_label=dataset_label,
        metric=metric,
        distance_label="interbin",
    ))
interbin_dict = multimodel_interbin.compute(activations_dict)

In [ ]:
fig = multimodel_interbin.plot(
    interbin_dict,
    title="Inter-bin distance",
)